# Day 34: Class Design and Composition

**Today's question:** real things contain other things: a playlist has songs, a course has students, a shop has
products. How do we build classes that work **together**, and how do we test them?

Before running a cell marked 🤔, predict what it will show.

## 1. A small class first: `Song`

Each song knows its own data and how to describe itself. `:02d` pads a number with a zero to 2 digits (`5` → `05`).

In [ ]:
class Song:
    def __init__(self, title, artist, seconds):
        if seconds <= 0:
            raise ValueError(f"A song must be longer than 0 seconds, got {seconds}.")
        self.title = title
        self.artist = artist
        self.seconds = seconds

    def duration(self):
        return f"{self.seconds // 60}:{self.seconds % 60:02d}"

    def __str__(self):
        return f"{self.title} by {self.artist} ({self.duration()})"

In [ ]:
chill_song = Song("Blue Hour", "Mei Lin", 215)
print(chill_song)
print(Song("Short Intro", "Diego R.", 65).duration())

## 2. Has-a: a `Playlist` holds `Song` objects

**Composition:** one object holds other objects in a list attribute. *A playlist **has** songs.*

In [ ]:
class Playlist:
    def __init__(self, name):
        self.name = name
        self.songs = []              # will hold Song objects

    def add(self, song):
        self.songs.append(song)

    def total_seconds(self):
        return sum([song.seconds for song in self.songs])

    def __str__(self):
        lines = [f"{self.name} ({len(self.songs)} songs)"]
        for number, song in enumerate(self.songs, start=1):
            lines.append(f"{number}. {song}")        # uses Song.__str__
        return "\n".join(lines)

In [ ]:
study_mix = Playlist("Study Mix")
study_mix.add(chill_song)
study_mix.add(Song("Rain on Tin", "Omar Haddad", 184))
study_mix.add(Song("Late Bus", "Zara K.", 247))
print(study_mix)
print("Total seconds:", study_mix.total_seconds())

The playlist never formats a duration itself: it asks each song (`f"{song}"` calls `Song.__str__`).

🤔 **Predict:** we change `chill_song` *after* adding it. Does the playlist's total change?

In [ ]:
chill_song.seconds = 200
print("Total seconds:", study_mix.total_seconds())

Yes: the list holds the **same object**, not a copy (Day 07 name tags). `chill_song` and `study_mix.songs[0]` are two
ways to reach one song.

## 3. Finding and removing: returning `None`

In [ ]:
class Playlist:
    def __init__(self, name):
        self.name = name
        self.songs = []

    def add(self, song):
        self.songs.append(song)

    def total_seconds(self):
        return sum([song.seconds for song in self.songs])

    def find(self, title):
        """Return the song with this title, or None if there is none."""
        for song in self.songs:
            if song.title.lower() == title.lower():
                return song
        return None                  # only reached when no song matched

    def remove(self, title):
        song = self.find(title)
        if song is None:
            raise ValueError(f"No song called {title} in {self.name}.")
        self.songs.remove(song)

    def longest(self):
        if not self.songs:
            return None
        longest_song = self.songs[0]
        for song in self.songs:
            if song.seconds > longest_song.seconds:
                longest_song = song
        return longest_song

    def __str__(self):
        lines = [f"{self.name} ({len(self.songs)} songs)"]
        for number, song in enumerate(self.songs, start=1):
            lines.append(f"{number}. {song}")
        return "\n".join(lines)

In [ ]:
road_trip = Playlist("Road Trip")
road_trip.add(Song("Open Road", "Fatima Noor", 198))
road_trip.add(Song("Desert Lights", "Ken Ito", 263))
road_trip.add(Song("Last Exit", "Sara B.", 175))

print(road_trip.find("desert lights"))
print(road_trip.find("Nothing Here"))
print("Longest:", road_trip.longest())

- `find` is a **question**: "not found" is a normal answer, so it returns `None`.
- `remove` is a **command**: removing a missing song is a mistake, so it raises.
- **`is None`** is the usual way to check for `None` (use `is` only for `None`; keep `==` for everything else).

🤔 **Predict:** what happens here?

In [ ]:
road_trip.remove("Nothing Here")

In [ ]:
road_trip.remove("open road")
print(road_trip)

## 4. Who does what?

| Job | Responsible class | Why |
|-----|-------------------|-----|
| format a song's length as `m:ss` | `Song` | needs only the song's own data |
| total length | `Playlist` | needs all songs |
| find a song by title | `Playlist` | owns the list |
| reject a song of 0 seconds | `Song.__init__` | each object protects its own state |

**Rule:** put a method in the class that owns the data it needs.

## 5. Designing from a description

> *A gym runs **classes** such as yoga or spinning. Each class has a **name**, a **coach** and a maximum number of
> **places**. **Members** have a **name** and a **membership number**. A member can **book** a place in a class
> and **cancel** it. The gym wants to **see who is booked** and **how many places are left**.*

| Noun / verb | Decision |
|-------------|----------|
| class (gym class), member | **classes**: own data *and* behaviour |
| name, coach, places, membership number | **attributes**: simple values |
| book, cancel | **methods** of the gym class (it owns the bookings) |
| see who is booked, places left | **question methods** |
| a gym class *has* booked members | **list attribute** `self.booked = []` |

Design first, as a **skeleton**: the method headers and docstrings, with `pass` as the body. It already runs.

In [ ]:
class Member:
    def __init__(self, name, number):
        self.name = name
        self.number = number


class GymClass:
    def __init__(self, name, coach, places):
        self.name = name
        self.coach = coach
        self.places = places
        self.booked = []             # Member objects

    def book(self, member):
        """Add a member. Raise ValueError if the class is full or the member is already booked."""
        pass

    def cancel(self, number):
        """Remove the member with this membership number. Raise ValueError if not booked."""
        pass

    def places_left(self):
        """Return how many places are still free."""
        pass

    def booked_names(self):
        """Return a list of the booked members' names."""
        pass


yoga = GymClass("Sunrise Yoga", "Asha", 12)
print(yoga.name, yoga.places_left())

`places_left()` gives `None` because the body is still `pass`. Next step: fill in one method at a time and **test
each one**. Tomorrow's project starter looks exactly like this.

## 6. Testing classes

A test function: **build** a fresh object, **act** (call a method), **assert** on the result or the state.
`pytest.raises` checks that an error happens (Day 29).

In [ ]:
import pytest


def test_total_seconds():
    mix = Playlist("Test")
    mix.add(Song("A", "X", 100))
    mix.add(Song("B", "Y", 50))
    assert mix.total_seconds() == 150


def test_find_ignores_case():
    mix = Playlist("Test")
    mix.add(Song("Late Bus", "Zara K.", 247))
    assert mix.find("LATE BUS").artist == "Zara K."
    assert mix.find("Early Bus") is None


def test_remove_unknown_song_raises_and_changes_nothing():
    mix = Playlist("Test")
    mix.add(Song("A", "X", 100))
    with pytest.raises(ValueError):
        mix.remove("Nothing")
    assert len(mix.songs) == 1


def test_song_rejects_zero_seconds():
    with pytest.raises(ValueError):
        Song("Silence", "Nobody", 0)

In a file called `test_playlist.py`, `uv run pytest` would find and run these automatically. In a notebook we call
them ourselves:

In [ ]:
test_total_seconds()
test_find_ignores_case()
test_remove_unknown_song_raises_and_changes_nothing()
test_song_rejects_zero_seconds()
print("all tests passed")

🤔 **Predict:** what does pytest do when the error we expect does **not** happen?

In [ ]:
with pytest.raises(ValueError):
    Song("Fine Song", "Someone", 120)

`Failed: DID NOT RAISE`: the test fails, which is exactly what we want if validation is missing.

## 7. Common mistakes

In [ ]:
# Mistake 1: adding a string instead of a Song object
mix = Playlist("Oops")
mix.add("Dreams")
print(mix.total_seconds())

Mistake 2: printing the list itself shows the objects' default text, because a list displays its items with a
different method (`__repr__`), not `__str__`. Loop instead:

In [ ]:
print(road_trip.songs)
for song in road_trip.songs:
    print(song)

*(Aside: defining `__repr__` the same way as `__str__` would make the list print nicely too. We don't need it.)*

In [ ]:
# Mistake 3: forgetting to create the list in __init__
class Playlist2:
    def __init__(self, name):
        self.name = name

    def add(self, song):
        self.songs.append(song)


Playlist2("Broken").add(chill_song)

## 8. Optional preview: inheritance ("is-a")

*Optional: not needed for the project or the checkpoint.* Composition is **has-a** (a playlist has songs).
Inheritance is **is-a**: a savings account **is a** bank account, with something extra.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be more than 0.")
        self.balance += amount

    def __str__(self):
        return f"{self.owner}'s account: balance {self.balance:.2f}"


class SavingsAccount(BankAccount):       # gets everything BankAccount has...
    def add_interest(self, percent):     # ...plus one extra method
        self.deposit(self.balance * percent / 100)

In [ ]:
ken = SavingsAccount("Ken", 200)
ken.deposit(50)          # inherited from BankAccount
ken.add_interest(4)      # only savings accounts have this
print(ken)               # inherited __str__

Use inheritance only for a true "is-a". In this course we build with composition.

## Recap

1. **Composition (has-a):** an object holds other objects in a list attribute and asks them to do their own jobs.
2. Search methods return the object or `None` (check with `is None`); commands that cannot do their job raise.
3. Design: nouns → classes or attributes, verbs → methods, "has a" → list attribute. Write a skeleton first.
4. Test classes with small functions: build, act, assert; `pytest.raises` for errors.

**Next time:** Project 04, the Inventory Manager: a `Product` class, an `Inventory` that has products, a menu and tests.